In [1]:
%pip install pycocotools ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 8.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import random
import sys
import numpy as np
import pandas as pd
import torch

from functools import partial
from pathlib import Path
from pycocotools.coco import COCO
from torch.utils.data import DataLoader
from transformers import RTDetrImageProcessor

PROJECT_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"
DIRECTORY = INPUT_ROOT / COMPETITION

if not (DIRECTORY / "train" / "annotations_coco.json").is_file():
    data_roots = sorted({
        path.parent.parent
        for path in INPUT_ROOT.rglob("annotations_coco.json")
        if path.parent.name == "train"
    })

    DIRECTORY = data_roots[0]

INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"
INFO_VAL = DIRECTORY / "val" / "annotations_coco.json"
INFO_TEST = DIRECTORY / "test_images.csv"

required_scripts = {
    "data_prep.py",
    "dataset_helpers.py",
    "model_builders.py",
    "torch_model.py",
    "yolo26_model.py",
    "rdtdetr_model.py",
    "experiment_runner.py",
    "experiment_results.py",
}

script_dirs = sorted({
    path.parent for path in INPUT_ROOT.rglob("rdtdetr_model.py")
    if path.parent.name == "scripts"
    and all(
        (path.parent / filename).is_file()
        for filename in required_scripts
    )
})

scripts_parent = str(script_dirs[0].parent)
if scripts_parent not in sys.path:
    sys.path.insert(0, scripts_parent)

from scripts.data_prep import load_annotations
from scripts.dataset_helpers import DetectionDataset, RTDetrCollator
from scripts.model_builders import build_rtdetr
from scripts.experiment_runner import run_rtdetr
from scripts.experiment_results import model_training_results_final

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
EPOCHS = 8
BATCH_SIZE = 2
FRAME_STEP = 3
CHECKPOINT = "PekingU/rtdetr_r50vd"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
processor = RTDetrImageProcessor.from_pretrained(CHECKPOINT)

coco_train = COCO(str(INFO_TRAIN))
coco_valid = COCO(str(INFO_VAL))
train_frames, train_boxes = load_annotations(coco_train, split="train")
valid_frames, valid_boxes = load_annotations(coco_valid, split="val")

train_annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
valid_annotations_by_image = {image_id: group for image_id, group in valid_boxes.groupby("image_id")}

sampled_train_frames = (train_frames.sort_values(["video", "frame_id"])
                        .groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

category_ids = sorted(coco_valid.getCatIds())
cat_to_label = {category_id: index for index, category_id in enumerate(category_ids)}
id_to_label = {index: coco_valid.cats[category_id]["name"] for category_id, index in cat_to_label.items()}
label_to_id = {name: index for index, name in id_to_label.items()}

preprocessor_config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

loading annotations into memory...
Done (t=6.29s)
creating index...
index created!
loading annotations into memory...
Done (t=0.62s)
creating index...
index created!


In [5]:
train_dataset = DetectionDataset(frames=sampled_train_frames, directory=DIRECTORY,
    annotations_by_image=train_annotations_by_image,
    empty_annotations=train_boxes.iloc[:0].copy(), validation=False, 
    horizontal=True, vertical=False, contrast=True, saturation=True
)

valid_dataset = DetectionDataset(
    frames=valid_frames, directory=DIRECTORY,
    annotations_by_image=valid_annotations_by_image,
    empty_annotations=valid_boxes.iloc[:0].copy(), validation=True
)

collator = RTDetrCollator(processor, cat_to_label)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collator)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collator)

In [6]:
model_kwargs = {"CHECKPOINT": CHECKPOINT, "id_to_label": id_to_label,
                "label_to_id": label_to_id, "num_queries": 150}

train_kwargs = {"processor": processor, "train_loader": train_loader,
                "valid_loader": valid_loader, "annotations_path": INFO_VAL,
                "device": device, "seed": SEED, "epochs": EPOCHS,
                "lr": 1e-4, "weight_decay": 1e-4, "max_hours": 5}

In [7]:
aug_train_histories, aug_train_results = run_rtdetr(**train_kwargs, model_method=partial(build_rtdetr, **model_kwargs,
                            matcher_bbox_cost=7.0), output_dir=PROJECT_ROOT / "outputs" / "rtdetr_aug_train",
                            lr_scheduler=True, step_size=4)

loading annotations into memory...
Done (t=0.46s)
creating index...
index created!


config.json:   0%|          | 0.00/5.11k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  172MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/764 [00:00<?, ?it/s]

[transformers] RTDetrForObjectDetection LOAD REPORT from: PekingU/rtdetr_r50vd
Key                                                 | Status   |                                                                                        
----------------------------------------------------+----------+----------------------------------------------------------------------------------------
model.decoder.class_embed.{0, 1, 2, 3, 4, 5}.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([8])          
model.enc_score_head.weight                         | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80, 256]) vs model:torch.Size([8, 256])
model.decoder.class_embed.{0, 1, 2, 3, 4, 5}.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80, 256]) vs model:torch.Size([8, 256])
model.denoising_class_embed.weight                  | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81, 256]) vs model:torch.Size([9, 256])
mod

Loading and preparing results...
DONE (t=2.22s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=106.35s).
Accumulating evaluation results...
DONE (t=4.87s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.190
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.362
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.173
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.090
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.161
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.347
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.103
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.256
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.323
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

In [8]:
aug_train_results_wrap = model_training_results_final(PROJECT_ROOT / "outputs" / "rtdetr_aug_train")

Training history


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,17.859426,0.00010,0.189632,0.361983,0.089789
1,2,15.213238,0.00010,0.174743,0.339759,0.089948
2,3,14.433858,0.00010,0.188732,0.358286,0.087055
3,4,13.873146,0.00010,0.179524,0.349204,0.081197
4,5,12.884164,0.00001,0.186961,0.358383,0.079153



Best validation results — epoch 1


,mAP,AP50,AP_small
0,0.189632,0.361983,0.089789
